###  Create the Vector Search endpoint (if it doesn't already exist)

In [0]:
%pip install databricks-vectorsearch --quiet
dbutils.library.restartPython()


In [0]:
dbutils.widgets.text("catalog", "bootcamp_students")
dbutils.widgets.text("schema", "madgula_sirisha_capstone")
dbutils.widgets.text("pdf_volume_path", "/Volumes/bootcamp_students/madgula_sirisha_capstone/landing/ContractOfCarrirage/")
dbutils.widgets.text("vs_endpoint_name", "flight-tracker-vs-endpoint")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
pdf_volume_path = dbutils.widgets.get("pdf_volume_path")
vs_endpoint_name = dbutils.widgets.get("vs_endpoint_name")

mv_chunks_table = f"{catalog}.{schema}.rag_document_chunks"
chunks_table = f"{catalog}.{schema}.rag_document_chunks_published"
index_name = f"{catalog}.{schema}.rag_document_chunks_index"

In [0]:
# Unable to create vector search on top of materialized view in Databricks
# Publishing the materialized view to a plain delta table to enable vector search

spark.sql(f"""
    CREATE OR REPLACE TABLE {chunks_table}
    AS SELECT * FROM {mv_chunks_table}
""")
 
spark.sql(f"ALTER TABLE {chunks_table} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
 
print(f"Published {mv_chunks_table} -> {chunks_table}")

In [0]:
from databricks.vector_search.client import VectorSearchClient

vsc = VectorSearchClient()

existing_endpoints = [e["name"] for e in vsc.list_endpoints().get("endpoints", [])]
if vs_endpoint_name not in existing_endpoints:
    vsc.create_endpoint(name=vs_endpoint_name, endpoint_type="STANDARD")
    print(f"Creating endpoint {vs_endpoint_name} — this can take a few minutes.")
else:
    print(f"Endpoint {vs_endpoint_name} already exists.")

In [0]:
# MAGIC %md
# MAGIC ## 6. Create the Delta Sync Index
# MAGIC `embedding_source_column="chunk_text"` + `embedding_model_endpoint_name`
# MAGIC means Databricks computes embeddings automatically using a hosted
# MAGIC embedding model at index-sync time — the only embedding calls in this
# MAGIC entire pipeline happen here.
# MAGIC `databricks-gte-large-en` is a pay-per-token embedding endpoint
# MAGIC available by default; no separate serving-endpoint setup needed.

# COMMAND ----------

import time

EMBEDDING_MODEL_ENDPOINT = "databricks-gte-large-en"

# Wait for the endpoint to be ONLINE before creating the index
ep = vsc.get_endpoint(vs_endpoint_name)
ep_state = ep.get("endpoint_status", {}).get("state", "UNKNOWN")
while ep_state != "ONLINE":
    print(f"Endpoint state: {ep_state} — waiting 30s...")
    time.sleep(30)
    ep = vsc.get_endpoint(vs_endpoint_name)
    ep_state = ep.get("endpoint_status", {}).get("state", "UNKNOWN")
print(f"Endpoint {vs_endpoint_name} is ONLINE.")

try:
    index = vsc.create_delta_sync_index(
        endpoint_name=vs_endpoint_name,
        source_table_name=chunks_table,
        index_name=index_name,
        pipeline_type="TRIGGERED",  # static corpus — no need for continuous sync
        primary_key="chunk_id",
        embedding_source_column="chunk_text",
        embedding_model_endpoint_name=EMBEDDING_MODEL_ENDPOINT,
    )
    print(f"Creating index {index_name} — this can take several minutes for a first sync.")
except Exception as e:
    if "already exists" in str(e).lower():
        index = vsc.get_index(endpoint_name=vs_endpoint_name, index_name=index_name)
        print(f"Index {index_name} already exists — reusing it.")
    elif "not found" in str(e).lower() and "endpoint" in str(e).lower():
        # Orphaned index from a deleted endpoint — drop the stale UC entry and retry
        print(f"Stale index detected (references a deleted endpoint). Cleaning up...")
        spark.sql(f"DROP TABLE IF EXISTS {index_name}")
        index = vsc.create_delta_sync_index(
            endpoint_name=vs_endpoint_name,
            source_table_name=chunks_table,
            index_name=index_name,
            pipeline_type="TRIGGERED",
            primary_key="chunk_id",
            embedding_source_column="chunk_text",
            embedding_model_endpoint_name=EMBEDDING_MODEL_ENDPOINT,
        )
        print(f"Creating index {index_name} — this can take several minutes for a first sync.")
    else:
        raise